In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Somendar Das Resume 1 (1) (3).pdf', 'Somendar Das Resume 1 (1) (2).pdf', 'Somendar Das Resume 1 (1) (1).pdf', 'Somendar Das Resume 1 (1).pdf', 'Colab Notebooks', 'Business_Analysis.ipynb', 'Business_Analysis pdf.pdf', 'Output Data Structure1.xlsx', 'BLACKCOFFER (1).ipynb', 'Blackcoffer Files', 'eng-spa.ipynb', 'eng-fren.ipynb', 'DSI Somendar (2).pdf', 'DSI Somendar (1).pdf', 'DSI Somendar.pdf', 'mllm.pdf', 'QT1 (4).pdf', 'QT1 (3).pdf', 'QT1 (2).pdf', 'QT1 (1).pdf', 'QT1.pdf', 'Internship Report.gdoc', 'Untitled spreadsheet.gsheet', 'q2.gsheet', 'q1.gsheet', 'q3.gsheet', 'q4.gsheet', 'BITCOIN.xlsx', 'Buliding_damage_project']


In [3]:
for folder in os.listdir('/content/drive/MyDrive'):
    print(folder)

Somendar Das Resume 1 (1) (3).pdf
Somendar Das Resume 1 (1) (2).pdf
Somendar Das Resume 1 (1) (1).pdf
Somendar Das Resume 1 (1).pdf
Colab Notebooks
Business_Analysis.ipynb
Business_Analysis pdf.pdf
Output Data Structure1.xlsx
BLACKCOFFER (1).ipynb
Blackcoffer Files
eng-spa.ipynb
eng-fren.ipynb
DSI Somendar (2).pdf
DSI Somendar (1).pdf
DSI Somendar.pdf
mllm.pdf
QT1 (4).pdf
QT1 (3).pdf
QT1 (2).pdf
QT1 (1).pdf
QT1.pdf
Internship Report.gdoc
Untitled spreadsheet.gsheet
q2.gsheet
q1.gsheet
q3.gsheet
q4.gsheet
BITCOIN.xlsx
Buliding_damage_project


In [4]:
PROJECT_DIR = "/content/drive/MyDrive/Buliding_damage_project"

print(os.path.exists(PROJECT_DIR))

print(os.listdir(PROJECT_DIR))

True
['balanced_dataset.csv', 'images']


In [5]:
import os
import cv2
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

print("Imports Successful")

Imports Successful


In [6]:
BASE_DIR = "/content/drive/MyDrive/Buliding_damage_project"

CSV_PATH = f"{BASE_DIR}/balanced_dataset.csv"

IMAGE_DIR = f"{BASE_DIR}/images"

print(BASE_DIR)
print(CSV_PATH)
print(IMAGE_DIR)

/content/drive/MyDrive/Buliding_damage_project
/content/drive/MyDrive/Buliding_damage_project/balanced_dataset.csv
/content/drive/MyDrive/Buliding_damage_project/images


In [7]:
import os

print(os.path.exists(BASE_DIR))
print(os.path.exists(CSV_PATH))
print(os.path.exists(IMAGE_DIR))

True
True
True


In [8]:
df = pd.read_csv(CSV_PATH)

print(df.shape)

df.head()

(16956, 10)


,uid,label,disaster_type,pre_image,post_image,xmin,ymin,xmax,ymax,area
0,3288f4b7-fe43-4e68-b8c2-61433c62a211,destroyed,fire,santa-rosa-wildfire_00000162_pre_disaster.png,santa-rosa-wildfire_00000162_post_disaster.png,756,244,815,283,2301
1,b67dbfea-09f9-4c40-8276-9fb6281aa9b8,destroyed,tsunami,palu-tsunami_00000112_pre_disaster.png,palu-tsunami_00000112_post_disaster.png,645,960,687,1011,2142
2,48ff0e3b-60df-4395-9118-685b7dac4a1c,destroyed,fire,santa-rosa-wildfire_00000190_pre_disaster.png,santa-rosa-wildfire_00000190_post_disaster.png,813,291,876,315,1512
3,2948acb5-e9ee-40bb-87af-fdea2b3c3694,destroyed,fire,santa-rosa-wildfire_00000063_pre_disaster.png,santa-rosa-wildfire_00000063_post_disaster.png,182,825,227,874,2205
4,a0ee2088-ddac-462d-812b-6b2061a627e2,destroyed,fire,santa-rosa-wildfire_00000079_pre_disaster.png,santa-rosa-wildfire_00000079_post_disaster.png,453,443,534,536,7533


In [9]:
label_map = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3
}

label_map

{'no-damage': 0, 'minor-damage': 1, 'major-damage': 2, 'destroyed': 3}

In [10]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)

Train: (13564, 10)
Val: (3392, 10)


In [11]:
print(df.shape)

print(train_df.shape)

print(val_df.shape)

(16956, 10)
(13564, 10)
(3392, 10)


In [12]:
label_map = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3
}

label_map

{'no-damage': 0, 'minor-damage': 1, 'major-damage': 2, 'destroyed': 3}

In [13]:
def crop_building(
    image,
    xmin,
    ymin,
    xmax,
    ymax,
    padding=30
):

    xmin = max(0, xmin-padding)
    ymin = max(0, ymin-padding)

    xmax = min(
        image.shape[1],
        xmax+padding
    )

    ymax = min(
        image.shape[0],
        ymax+padding
    )

    crop = image[
        ymin:ymax,
        xmin:xmax
    ]

    return crop

In [14]:
from torchvision import transforms

transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize((224,224)),

    transforms.ToTensor()

])

In [15]:
class XBDDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):

        self.df = dataframe

        self.image_dir = image_dir

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        pre_path = os.path.join(
            self.image_dir,
            row["pre_image"]
        )

        post_path = os.path.join(
            self.image_dir,
            row["post_image"]
        )

        pre_img = cv2.imread(pre_path)

        post_img = cv2.imread(post_path)

        pre_img = cv2.cvtColor(
            pre_img,
            cv2.COLOR_BGR2RGB
        )

        post_img = cv2.cvtColor(
            post_img,
            cv2.COLOR_BGR2RGB
        )

        pre_crop = crop_building(
            pre_img,
            row["xmin"],
            row["ymin"],
            row["xmax"],
            row["ymax"]
        )

        post_crop = crop_building(
            post_img,
            row["xmin"],
            row["ymin"],
            row["xmax"],
            row["ymax"]
        )

        if self.transform:

            pre_crop = self.transform(
                pre_crop
            )

            post_crop = self.transform(
                post_crop
            )

        label = label_map[
            row["label"]
        ]

        return (
            pre_crop,
            post_crop,
            label
        )

In [16]:
train_dataset = XBDDataset(
    train_df,
    IMAGE_DIR,
    transform
)

val_dataset = XBDDataset(
    val_df,
    IMAGE_DIR,
    transform
)

print(len(train_dataset))
print(len(val_dataset))

13564
3392


In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2
)

In [18]:
pre_imgs, post_imgs, labels = next(
    iter(train_loader)
)

print(pre_imgs.shape)
print(post_imgs.shape)
print(labels.shape)

torch.Size([8, 3, 224, 224])
torch.Size([8, 3, 224, 224])
torch.Size([8])


In [19]:
import torch
import torch.nn as nn
import torchvision.models as models

In [20]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [21]:
import time

start = time.time()

pre_imgs, post_imgs, labels = next(iter(train_loader))

end = time.time()

print("Time:", end - start)

Time: 21.448836088180542


In [22]:
!nvidia-smi

Thu Jun 25 11:48:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
class DualEfficientNet(nn.Module):

    def __init__(self, num_classes=4):

        super().__init__()

        backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        self.feature_extractor = backbone.features

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(

            nn.Linear(2560, 512),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(512, num_classes)

        )

    def forward(
        self,
        pre_img,
        post_img
    ):

        pre_feat = self.feature_extractor(
            pre_img
        )

        pre_feat = self.pool(
            pre_feat
        )

        pre_feat = torch.flatten(
            pre_feat,
            1
        )

        post_feat = self.feature_extractor(
            post_img
        )

        post_feat = self.pool(
            post_feat
        )

        post_feat = torch.flatten(
            post_feat,
            1
        )

        fused = torch.cat(
            [pre_feat, post_feat],
            dim=1
        )

        out = self.classifier(
            fused
        )

        return out

In [24]:
model = DualEfficientNet().to(device)

print(model.__class__.__name__)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 185MB/s]


DualEfficientNet


In [25]:
outputs = model(
    pre_imgs.to(device),
    post_imgs.to(device)
)

print(outputs.shape)

torch.Size([8, 4])


In [26]:
criterion = nn.CrossEntropyLoss()

In [27]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [28]:
def train_one_epoch():

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for pre_imgs, post_imgs, labels in train_loader:

        pre_imgs = pre_imgs.to(device)

        post_imgs = post_imgs.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            pre_imgs,
            post_imgs
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    loss = running_loss / len(train_loader)

    acc = 100 * correct / total

    return loss, acc

In [29]:
def validate():

    model.eval()

    running_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for pre_imgs, post_imgs, labels in val_loader:

            pre_imgs = pre_imgs.to(device)

            post_imgs = post_imgs.to(device)

            labels = labels.to(device)

            outputs = model(
                pre_imgs,
                post_imgs
            )

            loss = criterion(
                outputs,
                labels
            )

            running_loss += loss.item()

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    loss = running_loss / len(val_loader)

    acc = 100 * correct / total

    return loss, acc

In [30]:
EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc : {train_acc:.2f}%"
    )

    print(
        f"Val Loss  : {val_loss:.4f}"
    )

    print(
        f"Val Acc   : {val_acc:.2f}%"
    )

    print("-"*40)

Epoch 1/5
Train Loss: 0.7383
Train Acc : 70.91%
Val Loss  : 0.5830
Val Acc   : 77.48%
----------------------------------------
Epoch 2/5
Train Loss: 0.5391
Train Acc : 78.55%
Val Loss  : 0.5650
Val Acc   : 78.71%
----------------------------------------
Epoch 3/5
Train Loss: 0.4384
Train Acc : 82.45%
Val Loss  : 0.5364
Val Acc   : 80.10%
----------------------------------------
Epoch 4/5
Train Loss: 0.3547
Train Acc : 86.43%
Val Loss  : 0.6824
Val Acc   : 76.62%
----------------------------------------
Epoch 5/5
Train Loss: 0.2728
Train Acc : 89.54%
Val Loss  : 0.7000
Val Acc   : 78.12%
----------------------------------------


In [31]:
import torch

torch.save(
    model.state_dict(),
    "dual_efficientnet_final.pth"
)

print("Model Saved Successfully")

Model Saved Successfully


In [32]:
import shutil

shutil.copy(
    "dual_efficientnet_final.pth",
    "/content/drive/MyDrive/Buliding_damage_project/dual_efficientnet_final.pth"
)

print("Model Saved To Google Drive")

Model Saved To Google Drive


In [33]:
import os

print(os.path.exists(
    "/content/drive/MyDrive/Buliding_damage_project/dual_efficientnet_final.pth"
))

True


In [34]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for pre_imgs, post_imgs, labels in val_loader:

        pre_imgs = pre_imgs.to(device)
        post_imgs = post_imgs.to(device)

        outputs = model(pre_imgs, post_imgs)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Done!")

In [ ]:
class_names = [
    "no-damage",
    "minor-damage",
    "major-damage",
    "destroyed"
]

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names
    )
)

In [ ]:
class_names = [
    "no-damage",
    "minor-damage",
    "major-damage",
    "destroyed"
]

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()